# 第5回：転移学習・再学習・ニューラルネットワークモデルの紹介

この回は3つのパートで構成します：**ニューラルネットワークを試す ／ 転移学習を知る ／ モデルを運用する（永続化・監視・再学習）**。

**セルの動かし方**：各セル（灰色の枠）を選んで `Shift + Enter`（またはセル左の▷ボタン）を押すと実行できます。
**上から順に**実行してください。前のセルを飛ばすと、後のセルでエラーになります。

**AIと一緒に進める**：分からないコードは、セル全体ではなく気になる数行をM365 CopilotなどのAIへ貼り、
説明や修正を相談します。ただし、提案されたコードは必ず実行結果を見て確かめます。

まず「基本」と「演習」を進めます。「補足」は必要に応じて読み、
「発展（任意）」「追加演習（任意）」「自由課題（任意）」は飛ばしても構いません。


In [ ]:
# 【準備セル】教材フォルダの場所を自動で見つけます。中身は今は理解しなくてOK、そのまま実行してください。
from pathlib import Path

def find_repo_root(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("pyproject.tomlがある勉強会フォルダ内で実行してください")

ROOT = find_repo_root()
DATA = ROOT / "data"
print("教材フォルダ:", ROOT)


## この回で扱うこと

ニューラルネットワークを既存モデルと同条件で比較し、転移学習の考え方と限界を知り、モデルを運用・監視・再学習するところまで見据えます。

### 進め方

この回は3つのパートに分かれています。パート1から順に「基本」と「演習」を進めてください。
1日で終える必要はありません。「発展（任意）」と「追加演習（任意）」は、余裕がある場合だけ取り組みます。

### 用語について

初めて出る用語は、その用語を使うセルで説明します。ここでまとめて暗記する必要はありません。

> **実行前の30秒予想**：各パートの問いに、今の言葉で仮の答えを書いてから始めます。


---

# パート1：ニューラルネットワークを試す

**このパートの問い：複雑なモデルは、このデータでも必ず勝つのか。**


## ニューラルネットワークとは

**ニューラルネットワーク**は、入力を層状につないだ関数で表現を学習するモデルです。もっとも
基本的な形が**MLP（多層パーセプトロン）**で、入力層・**隠れ層**（中間の層）・出力層を重ねます。
画像やテキストなど、大量データがある分野で高い性能を出すことで知られています。

ここでの問いは、**このデータ（420行の表データ）でも、複雑なモデルは常に有利なのか**です。
第4回パート1で比較した木系モデルと、正面から同条件で比べます。


In [ ]:
import pandas as pd

df = pd.read_csv(DATA / "compound_experiments.csv")
print(f"{len(df)}行 × {len(df.columns)}列")
df.head()


## 演習：MLPと、これまでのモデルを同条件で比べる

`MLPClassifier`は数値の尺度に敏感なため、木系モデルと違い**標準化（StandardScaler）が必須**です。
同じ交差検証・同じ特徴量で、MLPとRandom Forestを並べます。


In [ ]:
from sklearn.model_selection import cross_validate, StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier

features = ["temperature_c", "reaction_time_h", "concentration_m", "molecular_weight", "logp", "tpsa"]
X = df[features]
y = df["active"]
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

candidates = {
    "MLP（隠れ層16）": make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), MLPClassifier(hidden_layer_sizes=(16,), max_iter=2000, random_state=42)),
    "Random Forest": make_pipeline(SimpleImputer(strategy="median"), RandomForestClassifier(n_estimators=200, max_depth=5, random_state=42)),
}
for name, est in candidates.items():
    result = cross_validate(est, X, y, cv=cv, scoring="f1")
    print(f"{name:16s} F1={result['test_score'].mean():.3f} ± {result['test_score'].std():.3f}")


### 出力の読み方

- **MLPがRandom Forestを上回るとは限りません**。420行という件数は、ニューラルネットワークが
  実力を発揮するには少なすぎることが多いのです。
- ばらつき（±）も見ます。MLPは初期値やデータの並びに敏感で、木系モデルよりばらつきが大きく
  出ることがあります。
- 「複雑なモデル＝高性能」ではなく、**データの量と質に見合ったモデルを選ぶ**という姿勢が大切です。


## 演習：尺度をそろえないとどうなるか

`StandardScaler`を抜いた場合と比べます。木系モデルは数値の尺度（桁の大きさ）に鈍感ですが、
MLPは内部で重みを掛け合わせるため、**尺度が違う列が混ざると学習が不安定になりやすい**という
性質があります。


In [ ]:
unscaled = make_pipeline(SimpleImputer(strategy="median"), MLPClassifier(hidden_layer_sizes=(16,), max_iter=2000, random_state=42))
scaled = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), MLPClassifier(hidden_layer_sizes=(16,), max_iter=2000, random_state=42))
for name, est in {"標準化なし": unscaled, "標準化あり": scaled}.items():
    result = cross_validate(est, X, y, cv=cv, scoring="f1")
    print(f"{name:8s} F1={result['test_score'].mean():.3f} ± {result['test_score'].std():.3f}")


### 出力の読み方

多くの場合、**標準化ありの方が安定して高いF1**になります。`molecular_weight`（数十〜百単位）と
`logp`（-1〜2程度）のように桁が大きく違う列が混ざると、尺度の大きい列に引きずられて学習が
うまく進まないことがある、という具体例です。


## 演習：ベースラインと比べる

第1回・第2回と同じ習慣で、**何もしないモデル（Dummy）**と**単純な線形モデル（Logistic回帰）**も
並べます。複雑なモデルの価値は、単純なモデルとの差でしか語れません。


In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression

baselines = {
    "多数派ベースライン": DummyClassifier(strategy="most_frequent"),
    "ロジスティック回帰": make_pipeline(SimpleImputer(strategy="median"), LogisticRegression(max_iter=1000)),
    "MLP（隠れ層16）": scaled,
    "Random Forest": candidates["Random Forest"],
}
for name, est in baselines.items():
    result = cross_validate(est, X, y, cv=cv, scoring="f1")
    print(f"{name:16s} F1={result['test_score'].mean():.3f}")


### 出力の読み方

MLPが多数派ベースラインより高ければ「何かは学習できている」と言えます。ただし**単純な
ロジスティック回帰にすら勝てない**なら、このデータ・この設定ではMLPを選ぶ理由がない、と
判断できます。


## まとめ

- MLPは入力を層状に処理するモデルで、**数値の標準化が必須**という点が木系モデルと異なる。
- 420行程度の表データでは、MLPが木系モデルに勝つとは限らない。
- モデルを複雑にする前に、**単純なモデルとの差を確認する**という第1回からの姿勢が、ここでも生きる。


## 発展（任意）：隠れ層の大きさと学習時間

隠れ層のユニット数を変えると、性能と学習時間がどう動くかを見ます。


In [ ]:
import time

rows = []
for units in [4, 16, 64, 128]:
    est = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), MLPClassifier(hidden_layer_sizes=(units,), max_iter=2000, random_state=42))
    start = time.perf_counter()
    result = cross_validate(est, X, y, cv=cv, scoring="f1")
    elapsed = time.perf_counter() - start
    rows.append({"隠れ層ユニット数": units, "F1": result["test_score"].mean(), "学習時間(秒)": round(elapsed, 2)})
pd.DataFrame(rows).round(3)


### 出力の読み方

ユニット数を増やすほど学習時間は伸びますが、F1が単調に良くなるとは限りません。**データ量に対して
モデルが複雑すぎる（過剰パラメータ）と、むしろ不安定になる**ことがあります。


### 早期終了（early stopping）を試す

`early_stopping=True`にすると、検証スコアの改善が止まった時点で学習を打ち切ります。過学習を防ぎつつ
学習時間を節約する工夫です。


In [ ]:
early = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), MLPClassifier(hidden_layer_sizes=(64,), max_iter=2000, early_stopping=True, random_state=42))
result = cross_validate(early, X, y, cv=cv, scoring="f1")
print(f"early_stopping=True: F1={result['test_score'].mean():.3f} ± {result['test_score'].std():.3f}")


### 出力の読み方

早期終了ありのF1を、上のセルの「隠れ層64」の結果と比べます。大きく変わらないなら、このデータでは
学習の打ち切りが結果に悪影響を与えていないということです。


### 活性化関数を変える

隠れ層の出力を非線形に変換する**活性化関数**を変えると、学習の挙動が変わります。既定の`relu`と
`tanh`を比べます。


In [ ]:
for activation in ["relu", "tanh"]:
    est = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), MLPClassifier(hidden_layer_sizes=(32,), activation=activation, max_iter=2000, random_state=42))
    result = cross_validate(est, X, y, cv=cv, scoring="f1")
    print(f"activation={activation:5s} F1={result['test_score'].mean():.3f} ± {result['test_score'].std():.3f}")


### 出力の読み方

差はデータやシードによって大小さまざまです。**どちらが常に優れているというものではなく**、
複数試して比較すること自体が、ニューラルネットワークを扱う上で必要な手間だと分かります。


### 学習曲線（loss_curve_）を見る

`MLPClassifier`は学習中の損失（誤差）の推移を`loss_curve_`に記録しています。学習がきちんと
収束しているかを確認できます。


In [ ]:
fitted = scaled.fit(X, y)
mlp_step = fitted.named_steps["mlpclassifier"]
print("学習回数（イテレーション数）:", len(mlp_step.loss_curve_))
print("最終損失:", round(mlp_step.loss_curve_[-1], 4))
print("最初の損失:", round(mlp_step.loss_curve_[0], 4))


### 出力の読み方

最終損失が最初の損失より十分小さければ、学習は進んでいます。**イテレーション数が`max_iter`の
上限に張り付いている**場合は、学習が収束しきっていない可能性があるので、`max_iter`を増やすか
`early_stopping`を検討します。


## 追加演習（任意）

回帰タスク（収率`yield_pct`の予測）でも、MLPRegressorとRandom Forestを比較します。90分の外の
自習向けです。


In [ ]:
from sklearn.neural_network import MLPRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold

reg_cv = KFold(n_splits=5, shuffle=True, random_state=42)
reg_candidates = {
    "MLPRegressor": make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), MLPRegressor(hidden_layer_sizes=(32,), max_iter=3000, random_state=42)),
    "Random Forest": make_pipeline(SimpleImputer(strategy="median"), RandomForestRegressor(n_estimators=200, max_depth=6, random_state=42)),
}
for name, est in reg_candidates.items():
    result = cross_validate(est, df[features], df["yield_pct"], cv=reg_cv, scoring="neg_mean_absolute_error")
    print(f"{name:14s} MAE={-result['test_score'].mean():.3f}")


### 出力の読み方

分類と同じ傾向が出るか確認します。**回帰でもMLPが必ず勝つわけではない**ことが多いはずです。
第3回パート1のMAEと見比べ、複雑なモデルを試す前に単純なモデルとの差を確認する習慣を続けます。


### L2正則化（alpha）を変える

`alpha`は重みの大きさを罰する正則化の強さです。大きくすると過学習を抑えますが、強すぎると
学習不足になります。


In [ ]:
for alpha in [0.0001, 0.01, 1.0]:
    est = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), MLPClassifier(hidden_layer_sizes=(32,), alpha=alpha, max_iter=2000, random_state=42))
    result = cross_validate(est, X, y, cv=cv, scoring="f1", return_train_score=True)
    print(f"alpha={alpha:<7} 学習F1={result['train_score'].mean():.3f}  検証F1={result['test_score'].mean():.3f}")


### 出力の読み方

`alpha`が小さいほど学習F1は高くなりやすい（覚え込みやすい）ですが、検証F1が伸びなければ過学習の
サインです。第1回で見た「木の深さと過学習」と同じ構図が、MLPでも`alpha`という別のダイヤルで
起こります。


### 学習時間をRandom Forestと比べる

MLPと木系モデルでは、学習にかかる時間の性質も異なります。同じデータで学習時間を比較します。


In [ ]:
import time

for name, est in {"MLP（隠れ層32）": make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), MLPClassifier(hidden_layer_sizes=(32,), max_iter=2000, random_state=42)), "Random Forest": RandomForestClassifier(n_estimators=200, max_depth=5, random_state=42)}.items():
    start = time.perf_counter()
    est.fit(X.fillna(X.median()), y)
    print(f"{name:16s} 学習時間={time.perf_counter() - start:.3f}秒")


### 出力の読み方

このデータ規模ではどちらも数秒以内に収まりますが、木系モデルは並列化がしやすく、データが
大きくなっても比較的速く学習できる傾向があります。速度も、モデルを選ぶ際の判断材料の1つです。


### 最適化アルゴリズム（solver）を変える

重みを更新する最適化アルゴリズムにも選択肢があります。既定の`adam`と、小規模データ向けとされる
`lbfgs`を比べます。


In [ ]:
for solver in ["adam", "lbfgs"]:
    est = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), MLPClassifier(hidden_layer_sizes=(32,), solver=solver, max_iter=2000, random_state=42))
    result = cross_validate(est, X, y, cv=cv, scoring="f1")
    print(f"solver={solver:6s} F1={result['test_score'].mean():.3f} ± {result['test_score'].std():.3f}")


### 出力の読み方

`lbfgs`は小規模データで安定しやすいとされますが、必ず勝つわけではありません。**ハイパーパラメータの
選択肢は多く、どれが良いかはデータ次第**という感覚を持ち帰ってください。


---

# パート2：転移学習を知る

**このパートの問い：少ないデータしかないとき、他所で学んだ知識を借りられないか。**


## 転移学習とは

**転移学習**は、大量データで先に学習しておいたモデル（**事前学習済みモデル**）を、手元の少ない
データで追加学習（**ファインチューニング**）して使う手法です。画像認識やテキスト処理の分野で
広く使われています。

考え方はシンプルです。「ゼロから学ぶより、既に近い分野を学んだモデルを土台にする方が、
少ないデータでも良い結果が出やすい」というものです。


## なぜこの教材では手を動かさないのか

この教材の`compound_experiments.csv`は420行の表データです。転移学習が効果を発揮するには、
**事前学習に使える大規模なデータと、それに近い領域の事前学習済みモデル**が必要ですが、この
規模の表データ単体では、事前学習を自分たちで行うことは現実的ではありません。

そのため、この回はコードを書かず、**考え方と実例を知ること**に絞ります。転移学習を学ぶ価値が
無いという意味ではなく、**「今回のデータでは前提条件が揃っていない」**という判断そのものが、
実務で重要な感覚です。


## 化学・創薬分野での実例

- **分子表現学習**：大量の分子構造（SMILES）から、分子の性質を数値ベクトルとして事前学習するモデル群。手元の少ない実験データでファインチューニングし、新しい予測タスクに使う。
- **画像ベースの実験スクリーニング**：顕微鏡画像や結晶写真などを対象に、大規模画像データセットで事前学習したモデルを土台に、少数の実験画像で追加学習する。
- **言語モデルの応用**：論文や特許テキストを大量に学習した言語モデルを、社内文書の分類・要約にファインチューニングする。

共通しているのは、**事前学習の領域と、手元データの領域が近いほど効果が出やすい**という点です。


## まとめ

- 転移学習＝事前学習済みモデルを、手元の少ないデータでファインチューニングして使う手法。
- この教材のデータ規模・形式では、事前学習を自分たちで行うことは現実的でない。
- 化学・創薬分野でも、分子表現学習や画像スクリーニングなど、条件が揃えば有効な場面がある。
- 「使えるかどうかを見極める」判断力も、手法そのものと同じくらい大切。


## 発展（任意）：事前学習済みモデルを探す観点

自分の業務データに転移学習が使えそうか検討するときの、確認ポイントを整理します。
コードは書かず、考え方の整理です。


### 確認する3つの観点

1. **領域の近さ**：事前学習に使われたデータと、自分のデータはどれくらい近い分野か。
2. **データ形式**：画像・テキスト・分子構造など、事前学習済みモデルが対応する形式に合っているか。
3. **ライセンスと利用条件**：商用利用の可否、社内データを外部サービスへ送ってよいか（機密情報の
   取り扱い）を必ず確認する。

この3点が揃わない場合、転移学習より、この教材で扱ってきたような**表データ向けの手法（回帰・分類・
特徴量エンジニアリング）**の方が、現実的な選択肢になることが多いです。


## 追加演習（任意）

自分の業務に関連しそうな事前学習済みモデルや論文を1つ調べ、次の3点を1〜2行ずつメモします。
90分の外の自習向けです。

1. どんなデータで事前学習されているか
2. 自分の業務データとどれくらい領域が近いか
3. 試すとしたら、最初にどんな小さな検証をするか


---

# パート3：モデルを運用する（永続化・監視・再学習）

**このパートの問い：モデルを「作って終わり」にしないために、運用で何をするか。**


## 運用のループ：学習 → 提供 → 監視 → 再学習

ここまでで「良いモデルを作る」ことはできました。実務では、そこからが本番です。モデルは
**作って終わりではなく、動かし続ける手順**まで扱います。

> **学習 → 提供（サービング）→ 監視 → 再学習 → …**

このループを回す考え方や道具をまとめて**MLOps**と呼びます。ここでは、保存（永続化）・説明書
（モデルカード）・適用範囲（適用領域）・監視・再学習の5つを扱います。


## 永続化：学習済みモデルをファイルに保存する

毎回学習し直すのは非効率で、再現性も損なわれます。`joblib`で学習済みPipelineを**丸ごと保存**し、
読み直しても**同じ予測**になることを`assert`で確かめます。前処理も一緒に保存される点が重要です。


In [ ]:
import joblib
import numpy as np
import pandas as pd
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

data = pd.read_csv(DATA / "compound_experiments.csv")
feat = ["temperature_c", "reaction_time_h", "concentration_m", "molecular_weight", "logp", "tpsa"]
X_tr, X_te, y_tr, y_te = train_test_split(data[feat], data["active"], test_size=0.25, random_state=42, stratify=data["active"])
final = make_pipeline(SimpleImputer(strategy="median"), RandomForestClassifier(n_estimators=200, max_depth=5, random_state=42)).fit(X_tr, y_tr)
path = ROOT / "workspace" / "final_model.joblib"
joblib.dump(final, path)
reloaded = joblib.load(path)
assert np.array_equal(final.predict(X_te), reloaded.predict(X_te)), "保存前後で予測が一致しません"
print("保存し読み直しても同じ予測:", path)


### 読みどころ

`assert`が通り「同じ予測」と出れば、保存→配布→再利用の流れが安全に回ることの確認になります。
`Pipeline`ごと保存するので、**受け取った人は前処理を意識せず`predict`するだけ**。第3回パート3でPipelineに
まとめた恩恵がここで効きます。


## 提供（サービング）：学習済みモデルを「関数」として使えるようにする

運用では、新しい試料が来るたびに学習し直しません。**保存済みモデルを読み込み、予測だけを返す
関数**を用意します。


In [ ]:
def predict_activity(samples):
    "新しい試料(DataFrame)へ、活性の予測(0/1)と確率を返す推論関数。"
    proba = reloaded.predict_proba(samples[feat])[:, 1]
    return pd.DataFrame(
        {"活性予測": (proba >= 0.5).astype(int), "活性確率": proba.round(3)},
        index=samples.index,
    )

display(predict_activity(X_te.head()))


### 出力の読み方

前処理ごと保存したPipelineなので、受け取った人は`predict_activity(新しいデータ)`を呼ぶだけで
予測できます。これが「サービング」の最小形です。Webサービスやバッチ処理も、裏でこの関数を
呼んでいるだけ、とイメージしてください。


## モデルカード：使い方の説明書を関数で作る

モデルは「精度の数字」だけ渡してもトラブルの元です。**誰向けか・何を決めるためか・限界・禁止事項**を
1枚にまとめた**モデルカード**を、関数で自動生成します。第2回パート2の問題設定が、そのまま説明書になります。


In [ ]:
from sklearn.metrics import f1_score

def build_model_card(name, estimator, X_valid, y_valid, notes) -> pd.DataFrame:
    "モデルの用途と評価をまとめた1枚のカードを作る。"
    pred = estimator.predict(X_valid)
    items = {
        "モデル名": name,
        "検証F1": round(f1_score(y_valid, pred), 3),
        "想定利用者": notes["利用者"],
        "支援する判断": notes["判断"],
        "既知の限界": notes["限界"],
        "使ってはいけない条件": notes["禁止"],
    }
    return pd.DataFrame({"項目": list(items), "内容": list(items.values())})

build_model_card("活性スクリーナ", reloaded, X_te, y_te, {
    "利用者": "実験担当者", "判断": "追試する候補の優先順位",
    "限界": "新規scaffoldでは精度低下の可能性", "禁止": "測定後の列を入力に使うこと",
})


### 読みどころ

出来上がったカードには、性能（F1）と**使う上での注意**が並びます。特に「使ってはいけない条件（測定後の
列を入力にしない）」は、第2回パート2〜3で扱ったリークの注意点です。**精度より先に限界を書く**のが、信頼される
モデル提供者の作法です。


## 監視と再学習：いつモデルを作り直すか

運用後は、次を定期的に見張ります。

- **入力のドリフト**：入力分布が学習時とずれていないか
- **予測の傾向**：予測の陽性率が急に変わっていないか
- **性能**：正解ラベルが遅れて届いたら、F1などを計算し直す
- **適用領域**：学習データから遠い入力が増えていないか（次のセクションで扱う）

これらが目安を超えたら**再学習のトリガー**です。新しいデータを足して学習し直し、**同じ検証
（第2回パート3）・同じ評価（第3回）で前のモデルと比較**してから入れ替えます。作って終わりにせず、
このループを回し続けることが、実データでモデルを役立て続けるコツです。


## まとめ

- **永続化**：Pipelineごと保存すれば、前処理を含めて復元できる。
- **サービング**：保存済みモデルを関数として公開すれば、使う側は前処理を意識しなくてよい。
- **モデルカード**：性能より先に「使ってよい範囲・使ってはいけない条件」を書く。
- **監視と再学習**：正解ラベルが無くても入力ドリフトは検知できる。再学習後は必ず旧モデルと比較する。


## 発展（任意）：適用領域とドリフトの検知

発展として、**適用領域**（予測してよい範囲）と、それを使った**ドリフト検知**を扱います。


### 適用領域：予測してよい範囲を数値化する

モデルは、学習データと似た試料には強いですが、かけ離れた試料では当てになりません。学習データからの
**近傍距離**を測り、遠すぎる（範囲外の）試料を「要確認」に自動で仕分けます。95%点を閾値にします。


In [ ]:
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

train_filled = X_tr.fillna(X_tr.median())
scaler = StandardScaler().fit(train_filled)
nn = NearestNeighbors(n_neighbors=5).fit(scaler.transform(train_filled))
train_dist = nn.kneighbors(scaler.transform(train_filled))[0].mean(axis=1)
threshold = np.quantile(train_dist, 0.95)
valid_dist = nn.kneighbors(scaler.transform(X_te.fillna(X_tr.median())))[0].mean(axis=1)
out_of_domain = valid_dist > threshold
print(f"適用領域外と判定された検証試料: {int(out_of_domain.sum())} / {len(valid_dist)} 件")
print("範囲外は予測を鵜呑みにせず、要確認に回す運用が考えられる。")


### 読みどころ

範囲外と判定された試料は、予測を鵜呑みにせず人が確認する。これが**安全にAIを使う**ということです。


### ドリフトを模擬する：入力がずれたら「監視」で気づけるか

運用後、測定装置のずれなどで入力分布が変わる（ドリフト）ことがあります。テストの温度を+20℃ずらし、
**正解ラベルが無くても異常に気づけるか**を確かめます。運用中は正解（活性の実測）がすぐには手に入らない
ため、F1のような指標は即座には測れません。だからこそ、正解なしで検知できる監視が重要になります。


In [ ]:
import numpy as np
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score

drift = X_te.copy()
drift["temperature_c"] = drift["temperature_c"] + 20

# (1) 正解が無くても分かる変化：予測の陽性率
print(f"予測の陽性率: {reloaded.predict(X_te).mean():.3f} → {reloaded.predict(drift).mean():.3f}")

# (2) 監視：元データとドリフト後を見分けられるか（adversarial validation, 第2回パート3）
cols = X_te.columns.tolist()
combined = pd.concat([X_te.assign(is_drift=0), drift.assign(is_drift=1)], ignore_index=True)
filled = combined[cols].fillna(combined[cols].median())
auc = cross_val_score(RandomForestClassifier(n_estimators=200, random_state=42), filled, combined["is_drift"], cv=5, scoring="roc_auc").mean()
print(f"監視AUC: {auc:.3f}（0.5=変化なし / 1.0に近い=明確な分布変化）")

# 参考：正解が手に入ればF1でも確認できる（運用中は正解が遅れて届く）
print(f"参考F1: {f1_score(y_te, reloaded.predict(X_te)):.3f} → {f1_score(y_te, reloaded.predict(drift)):.3f}")


### 出力の読み方

- **監視AUCが0.5をはっきり上回る**なら、元データとドリフト後をモデルが見分けられる＝入力分布が変化した、という警報です。温度を+20℃ずらしたので、AUCは0.5より明確に高く出るはずです（1に近いほど変化が大きい）。
- **予測の陽性率**の変化も、正解ラベル無しで「何かが変わった」と気づける手がかりです。
- 一方、**参考F1は運用中すぐには測れません**（正解が遅れて届くため）。しかもこのデータ・特徴量では変化が小さく、性能指標だけに頼ると見逃しかねません。だからこそ、正解なしで異常を検知するadversarial validation（第2回パート3）のような監視が実務で効きます。


## 追加演習（任意）

「渡せる成果物」を実際に書き出します。90分の外の自習向けです。まず**モデルカードをMarkdown＋JSONで
保存**し、第三者が読める形にします。


In [ ]:
import json

card = build_model_card("活性スクリーナ", reloaded, X_te, y_te, {
    "利用者": "実験担当者", "判断": "追試候補の優先順位",
    "限界": "新規scaffoldで精度低下の可能性", "禁止": "測定後の列を入力に使うこと",
})
lines = ["# モデルカード", ""]
for _, r in card.iterrows():
    lines.append(f"- **{r['項目']}**: {r['内容']}")
(ROOT / "workspace" / "model_card.md").write_text("\n".join(lines), encoding="utf-8")

meta = {"features": feat, "n_train": int(len(X_tr)), "model": "RandomForest(max_depth=5)"}
(ROOT / "workspace" / "model_meta.json").write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8")
print("保存: workspace/model_card.md, workspace/model_meta.json")
print("\n".join(lines))


### 出力の読み方

`model_card.md`は人が読む説明書、`model_meta.json`は機械が読む来歴（使った特徴量・学習件数・モデル種別）。
モデルと一緒にこの2つを残すと、**半年後の自分や引き継ぎ先が再現・判断できます**。


### 成績表をファイルに書き出す

`classification_report`を表として保存します。引き継ぎに添付できる、機械可読な成績表です。


In [ ]:
from sklearn.metrics import classification_report

rep = classification_report(y_te, reloaded.predict(X_te), target_names=["非活性", "活性"], output_dict=True)
rep_df = pd.DataFrame(rep).T.round(3)
rep_df.to_csv(ROOT / "workspace" / "classification_report.csv")
display(rep_df)


### 出力の読み方、そしてこの教材の終わりに

クラスごとのprecision/recall/F1と全体のaccuracyが表になり、CSVで保存されます。数字だけを渡すのではなく、
**モデルカード（用途と限界）＋メタ情報（来歴）＋成績表**をひとまとめに渡す。ここまでできれば、
「作って終わり」から「運用でき、引き継げる」モデルへの橋を渡せています。全5回、おつかれさまでした。


---

## よくある誤り

- 複雑なモデル＝高性能だと思い込む
- スケーリングなしで数値特徴量をそのままMLPへ渡す
- 少量データでも層を増やせば良くなると考える
- 転移学習を使えば少ないデータでも必ず精度が上がると考える
- 事前学習の対象領域と手元データの領域が遠いのに流用する
- コードを書かずに概念だけで「使ったつもり」になる
- モデル単体だけ保存し、前処理を保存し忘れる
- 運用後に何も監視せず放置する
- 再学習したモデルを、比較せずにそのまま入れ替える

## 自習（任意・30〜60分）

- 隠れ層のユニット数を変え、検証スコアと学習時間の変化を記録する
- MLPが木系モデルに負けた理由を、データ件数の観点から1文で書く
- 自分の業務データに近い分野で、事前学習済みモデルが公開されていないか調べる
- 分子表現学習モデルの論文・紹介記事を1つ読み、要点を3行で書く
- 保存したPipelineを読み直し、同じ入力で同じ予測になるか検証する
- 適用領域スコアを閾値化し、範囲外の試料を要確認として仕分ける

成果は完成したコードでなくても、予想・変更点・出力・解釈を4行で残せば十分です。

## 振り返りチェック

1. MLPとRandomForestは何が違うか
2. このデータでMLPが必ず勝つとは限らない理由は何か
3. ニューラルネットワークが有利になりやすい条件は何か
4. 事前学習とファインチューニングの関係は何か
5. このデータで転移学習が使いにくい理由は何か
6. 化学・創薬分野での転移学習の実例を1つ挙げられるか
7. 永続化で何を一緒に保存すべきか
8. 運用後に監視すべき指標は何か
9. 再学習後、入れ替え前に何を確認すべきか

答えに詰まった項目が、次に見返す場所です。暗記ではなくNotebookの該当セルを指せればOKです。
